# Локальные нейросетевые и географические эксперименты

Основное воспроизводимое решение — `Avito.ipynb`. Этот notebook читает реальные
отчёты и позволяет повторить отдельные эксперименты из сопровождаемого кода
в `.development`. Каждый запуск работает локально, без inference API.

По умолчанию дорогое обучение выключено. Для повторения выбери соответствующие
значения `RUN_* = True`. Файлы Parquet должны лежать рядом с notebook.
`answer.csv` эти исследования не изменяют. Полный журнал ограничений —
`IMPROVEMENT_PROGRESS.md`; экспериментальный microcat-ответ воспроизводится
отдельным `Avito_microcat_candidate_v6.ipynb`.


In [1]:
from pathlib import Path
import json
import runpy
ROOT = Path.cwd()
assert (ROOT / 'train.parquet').exists(), 'Открой notebook в папке проекта'
RUN_CROSS_ENCODER = False
RUN_QUERY_ENCODER = False
RUN_QUERY_RANKER = False
RUN_SERVICE_GEOGRAPHY = False
RUN_COLD_SENSITIVITY = False
RUN_FINAL_QUERY_REFIT = False
RUN_BLEND_CHECK = False
RUN_QUERY_ERROR_ANALYSIS = False

def show_report(folder, filename):
    path = ROOT / 'artifacts' / folder / filename
    if path.exists():
        print(path.relative_to(ROOT))
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
    else:
        print('Эксперимент ещё не завершён:', path.relative_to(ROOT))

def run_local(name):
    # run_name='__main__' сохраняет совместимость локальных pickle/joblib-классов.
    runpy.run_path(str(ROOT / '.development' / name), run_name='__main__')


## Frozen cross-encoder

`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` совместно кодирует запрос и
объявление из первых 300/500 результатов v5. Проверяем только development:
готовый текстовый score может не соответствовать фактическому выбору услуги.
Модель скачивается один раз; ревизия и SHA256 проверяются downloader-ом.
Проверка прямой смеси не заменяет обучение признака внутри ранкера.


In [2]:
if RUN_CROSS_ENCODER:
    if not (ROOT/'models/mmarco-miniLM-cross-encoder/manifest.json').exists():
        run_local('download_cross_encoder.py')
    run_local('cross_encoder_pilot.py')
show_report('cross-encoder-pilot', 'pilot_report.json')


artifacts\cross-encoder-pilot\pilot_report.json
{
  "pilot_only": true,
  "fingerprint": "bb3d663d6b186022",
  "model": {
    "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "license": "Apache-2.0",
    "publisher_weights_sha256": "5daeca2481a76b5976a2bdc32f0a78532b6716da4f8cd3ff59460ef8d2f359b4",
    "transport_host": "https://hf-mirror.com",
    "sha256": {
      "config.json": "cc2cfe51aa3fd759d21d21acf5dfd6994aa67a3c9210636d22e143699d336c77",
      "tokenizer_config.json": "e7fbfbfa6347b4e414c1cee50d142e2c2f9a895dad68b068ae83a8b564c3837e",
      "special_tokens_map.json": "378eb3bf733eb16e65792d7e3fda5b8a4631387ca04d2015199c4d4f22ae554d",
      "tokenizer.json": "62c24cdc13d4c9952d63718d6c9fa4c287974249e16b7ade6d5a85e7bbb75626",
      "model.safetensors": "5daeca2481a76b5976a2bdc32f0a78532b6716da4f8cd3ff59460ef8d2f359b4",
      "README.md": "474736a65d6393a060119a8dc304563af67af4d8d86ccfee4a05dd0df107fc11"
  

## Query-encoder E5 с замороженными документами

Используем разрешённые пары, включая объявления вне benchmark-корпуса.
Контрастивные minibatch учитывают несколько известных положительных документов;
регуляризация к исходным query-векторам ограничивает дрейф. Пилот исключает
все тексты development и контроля из своей обучающей истории.
Подбор смеси идёт по development; затем параметры фиксируются для контроля.


In [3]:
if RUN_QUERY_ENCODER:
    run_local('query_encoder_pilot.py')
    run_local('query_encoder_control.py')
show_report('query-encoder-pilot', 'training_report.json')
show_report('query-encoder-pilot', 'pilot_report.json')
show_report('query-encoder-pilot', 'control_report.json')


artifacts\query-encoder-pilot\training_report.json
{
  "fingerprint": "2f9b3ff932e04986",
  "config": {
    "epochs": 3,
    "checkpoints": [
      1,
      3
    ],
    "batch_size": 128,
    "learning_rate": 2e-05,
    "weight_decay": 0.01,
    "temperature": 0.05,
    "teacher_weight": 0.2,
    "max_query_length": 64,
    "seed": 261831,
    "document_encoder_frozen": true,
    "negative_source": "in-batch",
    "multiple_positive_mask": true,
    "filters_in_encoder": false
  },
  "training_pairs_available": 379988,
  "training_query_texts": 69710,
  "document_vectors": 292210,
  "sampled_pairs": 209130,
  "development_text_overlap": 0,
  "document_vectors_sha256": "10c041daa3e5cf0b220e644a451636ea03622a2ceaa14b2058be348d654ed516",
  "original_encoder_weights_sha256": "1a55775f53449dac10a2bcbc312469fac40b96d53198c407081a831f81c98477",
  "gpu_peak_allocated_MiB": 4038.6,
  "logs": [
    {
      "epoch": 1,
      "mean_loss": 1.9330714328573384,
      "elapsed_seconds": 37.41
    },


## Обучаемый отбор с новым E5-сигналом

Шесть независимых query-encoder соответствуют трём фолдам в двух режимах
истории. Собственные контексты исключаются; в режиме новых текстов исключается
весь текст запроса. LambdaRank получает OOF-оценки, поэтому обучающая пара
не используется для обучения энкодера, который считает её признак.
Признаки: cosine, cosine с географией, изменения относительно исходной E5
и логарифмы рангов внутри полного пула. Группы и отрицательные примеры равны v5.


In [4]:
if RUN_QUERY_RANKER:
    run_local('query_encoder_rank.py')
show_report('query-encoder-rank', 'selection.json')
show_report('query-encoder-rank', 'control.json')
if RUN_BLEND_CHECK:
    run_local('query_micro_blend.py')
show_report('query-micro-blend', 'report.json')
if RUN_QUERY_ERROR_ANALYSIS:
    run_local('query_rank_error_analysis.py')
show_report('query-encoder-rank', 'error_summary.json')
if RUN_FINAL_QUERY_REFIT:
    run_local('export_query_encoder_candidate.py')
show_report('query-encoder-candidate', 'manifest.json')
show_report('query-encoder-candidate', 'portable_reproduction.json')


artifacts\query-encoder-rank\selection.json
{
  "winner": {
    "variant": "query_ranker",
    "trees": 400,
    "weight": 0.75,
    "matched_recall50": 0.9501708535101647,
    "unseen_macro_recall50": 0.9453676470588235,
    "actual_known_macro_recall50": 0.9530800821355236,
    "held_context_macro_recall50": 0.9466911764705882,
    "held_actual_known_fraction": 0.7161764705882353
  },
  "baseline": {
    "matched_recall50": 0.9476002382121281,
    "unseen_macro_recall50": 0.9434558823529412,
    "actual_known_macro_recall50": 0.9504106776180699,
    "held_context_macro_recall50": 0.9441911764705883,
    "held_actual_known_fraction": 0.7161764705882353
  },
  "fingerprint": "bf872e8b9abbb368",
  "control_used_for_selection": false,
  "feature_names": [
    "learned_cosine",
    "learned_geo",
    "learned_cosine_delta",
    "learned_geo_delta",
    "learned_log_cosine_rank",
    "learned_log_geo_rank"
  ],
  "limitation": "Evaluation encoder excludes all dev/control query texts; held-

## География по виду услуги

Оцениваем вероятность совпадения локации и переходов между локациями отдельно
для microcat. Малые выборки сглаживаются глобальной статистикой. Эти признаки
передаются ранкеру; жёсткого географического фильтра нет. OOF-история совпадает
с историей v5 и исключает собственные метки.


In [5]:
if RUN_SERVICE_GEOGRAPHY:
    run_local('service_geo_experiment.py')
show_report('service-geography-v6', 'selection.json')
show_report('service-geography-v6', 'control.json')


artifacts\service-geography-v6\selection.json
{
  "winner": {
    "variant": "micro",
    "trees": 200,
    "weight": 0.75,
    "matched_recall50": 0.9495279867465303,
    "unseen_macro_recall50": 0.9450735294117647,
    "actual_known_macro_recall50": 0.9536960985626284,
    "held_context_macro_recall50": 0.9453676470588235,
    "held_actual_known_fraction": 0.7161764705882353
  },
  "baseline": {
    "matched_recall50": 0.9476002382121281,
    "unseen_macro_recall50": 0.9434558823529412,
    "actual_known_macro_recall50": 0.9504106776180699,
    "held_context_macro_recall50": 0.9441911764705883,
    "held_actual_known_fraction": 0.7161764705882353
  },
  "fingerprint": "3a206d9460cea8d8",
  "control_used_for_selection": false,
  "feature_names": [
    "service_same_location_probability",
    "service_location_transition_probability",
    "service_log_support",
    "service_local_log_support",
    "service_geo_supported"
  ],
  "base_micro_selection": {
    "variant": "both",
    "tree

## Чувствительность к доле новых положительных объявлений

На фиксированных 400 development-контекстах изменяется доля собственных
положительных items, удаляемых из истории: 0%, 50%, 90%, 100%. Метки остаются
полными; веса ранкера заморожены. Это ограниченная диагностическая выборка,
она не доказывает независимость результата бенчмарка от cold items.


In [6]:
if RUN_COLD_SENSITIVITY:
    run_local('check_cold_sensitivity.py')
show_report('cold-sensitivity-v5', 'report.json')


artifacts\cold-sensitivity-v5\report.json
{
  "fingerprint": "26b52c75b6fe8fee",
  "frozen_model": "v5",
  "contexts": 400,
  "positions": [
    3,
    4,
    9,
    12,
    13,
    23,
    28,
    31,
    40,
    44,
    52,
    66,
    67,
    79,
    93,
    95,
    103,
    134,
    184,
    198,
    218,
    226,
    229,
    235,
    260,
    264,
    274,
    275,
    282,
    306,
    311,
    314,
    320,
    355,
    357,
    359,
    362,
    366,
    380,
    383,
    386,
    394,
    399,
    413,
    416,
    427,
    438,
    439,
    454,
    462,
    463,
    467,
    477,
    493,
    517,
    533,
    540,
    541,
    544,
    546,
    553,
    556,
    573,
    588,
    593,
    594,
    595,
    598,
    615,
    619,
    623,
    627,
    633,
    639,
    643,
    662,
    663,
    670,
    671,
    698,
    710,
    722,
    738,
    739,
    750,
    753,
    755,
    768,
    779,
    783,
    786,
    793,
    797,
    801,
    802,
    807,
    821,
    8